<span style="font-family: Arial; font-weight:bold;font-size:2.1em;color:#fff;">-----------------------------------
    
<span style="font-family: Arial; font-weight:bold;font-size:2.1em;color:#fa9200;">Testing out Indicators across various Stocks to Validate
    
<span style="font-family: Arial; font-weight:bold;font-size:2.1em;color:#fff;">-----------------------------------

# Import Packages

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from datetime import date, datetime, timedelta
import os
import re
import matplotlib as mpl
import matplotlib.pyplot as plt
from sklearn.neighbors import KNeighborsClassifier
import statsmodels.formula.api as sm
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
import yfinance as yf
from scipy.signal import argrelextrema
from collections import defaultdict
import sqlite3
import warnings
warnings.filterwarnings("ignore")
import Indicators
import Measurement
import Charts

# Grab Data for one Ticker

In [2]:
tickers = pd.read_csv('C:/Users/rschaub/Documents/SP500.csv', encoding='cp1252')
ticker = tickers['Symbol'][0]
#ticker = 'AMZN'
#ticker = 'WDC'
events = {'ihs_event':'bull','hs_event':'bear','fw_event':'bull','rw_event':'bear'}
df = Indicators.get_ticker(ticker,500)
df

,datetime,open,high,low,close,volume,dividends,stock splits,date,ticker
0,2023-11-01 09:30:00-04:00,76.220734,76.379265,75.426422,75.686371,322632,0.0,0.0,2023-11-01 09:30:00,MMM
1,2023-11-01 10:30:00-04:00,75.698738,75.775505,74.949829,75.117058,278400,0.0,0.0,2023-11-01 10:30:00,MMM
2,2023-11-01 11:30:00-04:00,75.100334,75.250839,74.995819,75.158859,207744,0.0,0.0,2023-11-01 11:30:00,MMM
3,2023-11-01 12:30:00-04:00,75.150505,75.250839,74.958191,74.991638,142573,0.0,0.0,2023-11-01 12:30:00,MMM
4,2023-11-01 13:30:00-04:00,74.991638,75.468231,74.942390,75.225754,159313,0.0,0.0,2023-11-01 13:30:00,MMM
...,...,...,...,...,...,...,...,...,...,...
3475,2025-10-29 11:30:00-04:00,167.110001,167.199997,166.500000,167.029999,191998,0.0,0.0,2025-10-29 11:30:00,MMM
3476,2025-10-29 12:30:00-04:00,167.009995,167.097504,166.639999,166.839996,131858,0.0,0.0,2025-10-29 12:30:00,MMM
3477,2025-10-29 13:30:00-04:00,166.839996,166.919998,165.160004,165.490005,286037,0.0,0.0,2025-10-29 13:30:00,MMM
3478,2025-10-29 14:30:00-04:00,165.410004,165.619995,163.809998,164.389999,463990,0.0,0.0,2025-10-29 14:30:00,MMM


# Add SMA, Min/Max, Events

In [3]:
SMAs = [5,30,60,90]
smoothing = 7
window = 7

df = Indicators.get_sma(df,SMAs)
events_df = {}
minmax = Indicators.get_max_min(df, smoothing, window)
for event in events:
    events_df[event] = getattr(Indicators,event)(minmax)
#hs = Indicators.find_HS(minmax).reset_index(drop=True)
#invhs = Indicators.find_IHS(minmax).reset_index(drop=True)
#rw = Indicators.find_RW(minmax,.03).reset_index(drop=True)
#fw = Indicators.find_FW(minmax,.03).reset_index(drop=True)
df

,datetime,open,high,low,close,volume,dividends,stock splits,date,ticker,SMA5,SMA30,SMA60,SMA90
0,2023-11-01 09:30:00-04:00,76.220734,76.379265,75.426422,75.686371,322632,0.0,0.0,2023-11-01 09:30:00,MMM,NaN,NaN,NaN,NaN
1,2023-11-01 10:30:00-04:00,75.698738,75.775505,74.949829,75.117058,278400,0.0,0.0,2023-11-01 10:30:00,MMM,NaN,NaN,NaN,NaN
2,2023-11-01 11:30:00-04:00,75.100334,75.250839,74.995819,75.158859,207744,0.0,0.0,2023-11-01 11:30:00,MMM,NaN,NaN,NaN,NaN
3,2023-11-01 12:30:00-04:00,75.150505,75.250839,74.958191,74.991638,142573,0.0,0.0,2023-11-01 12:30:00,MMM,NaN,NaN,NaN,NaN
4,2023-11-01 13:30:00-04:00,74.991638,75.468231,74.942390,75.225754,159313,0.0,0.0,2023-11-01 13:30:00,MMM,75.235936,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3475,2025-10-29 11:30:00-04:00,167.110001,167.199997,166.500000,167.029999,191998,0.0,0.0,2025-10-29 11:30:00,MMM,167.051373,168.593229,164.116703,160.148180
3476,2025-10-29 12:30:00-04:00,167.009995,167.097504,166.639999,166.839996,131858,0.0,0.0,2025-10-29 12:30:00,MMM,167.087000,168.469896,164.373870,160.330069
3477,2025-10-29 13:30:00-04:00,166.839996,166.919998,165.160004,165.490005,286037,0.0,0.0,2025-10-29 13:30:00,MMM,166.820999,168.325396,164.599087,160.510402
3478,2025-10-29 14:30:00-04:00,165.410004,165.619995,163.809998,164.389999,463990,0.0,0.0,2025-10-29 14:30:00,MMM,166.185999,168.138396,164.806253,160.683624


# Plot All Events

In [4]:
Charts.chart_all(df,events,events_df,SMAs,minmax)

In [6]:
df2 = Indicators.main(ticker,events,500,SMAs,smoothing,window)
eventType = 'rw_event'
ID = list(df2[df2.date==df2[df2[eventType]==1].date.min()].index)[0]
chartdf = df2.iloc[ID-10:ID+95].copy()
minmax2 = minmax[(minmax.date>=chartdf.date.min())&(minmax.date<=chartdf.date.max())].copy()
fig = go.Figure()
fig = go.Figure(data=[go.Candlestick(x=chartdf['date'],
                open=chartdf['open'],
                high=chartdf['high'],
                low=chartdf['low'],
                close=chartdf['close'],
                name='Price')])
for n in SMAs:
    rgb = 'rgb({},{},{})'.format(250-n*2,n*2,250-n*2)
    fig.add_trace(go.Scatter(x=chartdf['date'],
                y=chartdf['SMA{}'.format(n)],
                mode='lines',
                name='SMA{}'.format(n),
                line=dict(color=rgb)))
fig.add_trace(go.Scatter(x=minmax2['date'],
                y=minmax2['close'],
                mode='markers',
                name='Local Min/Max',
                marker=dict(color='rgb(200,200,0)',
                            size=10,
                            opacity=0.5)
                        ))
fig.update_xaxes(
        rangeslider_visible=True,
        rangebreaks=[
            # NOTE: Below values are bound (not single values), ie. hide x to y
            dict(bounds=["sat", "mon"]),  # hide weekends, eg. hide sat to before mon
            dict(bounds=[16, 9.5], pattern="hour"),  # hide hours outside of 9.30am-4pm
            #dict(values=["2024-12-25", "2024-01-01","2024-01-15","2024-02-19"])  # hide holidays (Christmas and New Year's, etc)
        ]
    )
fig.update_layout(title={'text':ticker, 'xanchor':'center', 'yanchor':'top','x':0.5},
                height=600,
                title_font={"family":"arial","color":"gray","size":38},
                paper_bgcolor='rgb(220,230,230)'
                 )
fig.show()

# Add Variables for Measurement Main Function

In [8]:
M = 80
K = 500
window = 7
smoothing = 7
events = {'ihs_event':'bull','hs_event':'bear','fw_event':'bull','rw_event':'bear'}
bound = 0.03
SMAs = [30,60,90]

### Run Measurement Main

In [9]:
df3, final4 = Measurement.main(events, SMAs, smoothing, window, M, K, bound)

$BRK.B: possibly delisted; no price data found  (period=500d) (Yahoo error = "No data found, symbol may be delisted")


'DataFrame' object has no attribute 'Datetime'


$BF.B: possibly delisted; no price data found  (period=500d)


'DataFrame' object has no attribute 'Datetime'


### Check df3 for one Event

In [10]:
event = 'fw_event'
df3.groupby(['ticker',event + '_start_time',event + '_end_time']).agg({'volume':'size',event:'mean'}).reset_index().head(5)

,ticker,fw_event_start_time,fw_event_end_time,volume,fw_event
0,A,2023-11-01 09:30:00,2024-04-08 09:30:00,746,0.0
1,A,2024-04-08 10:30:00,2024-04-16 14:30:00,47,1.0
2,A,2024-04-16 15:30:00,2025-02-25 11:30:00,1490,0.0
3,A,2025-02-25 12:30:00,2025-03-04 09:30:00,33,1.0
4,A,2025-03-04 10:30:00,2025-04-01 13:30:00,144,0.0


### Run Chart for One Event & Ticker

In [11]:
event = 'rw_event'
ticker = 'TXN'
    
Charts.chart_event(df3,event,ticker,SMAs)

### Final Result for Prefect/Slack

In [9]:
final4.sort_values(['event_end_time'],ascending=0).head(15)

,index,ticker,event,event_observations,event_start_time,event_end_time,after_event_observations,after_event_mean,after_event_start_time,after_event_end_time,count,mean,min,median,max,Indicator,event_count,event_success,stock_success
585,107,CHD,fw_event,63.0,2024-09-09 09:30:00,2024-09-19 15:30:00,1.0,0.009195,2024-09-19 15:30:00,2024-09-24 10:30:00,2682.0,0.032586,-0.004298,0.027248,0.112766,0,1.0,0.798283,1.0
162,280,ETN,rw_event,59.0,2024-09-09 12:30:00,2024-09-19 14:30:00,1.0,-0.001277,2024-09-19 14:30:00,2024-09-24 10:30:00,2540.0,-0.031236,-0.167012,-0.021451,0.033526,0,2.0,0.794776,1.0
265,489,LYB,rw_event,33.0,2024-09-13 09:30:00,2024-09-19 13:30:00,1.0,-0.013429,2024-09-19 13:30:00,2024-09-24 10:30:00,2949.0,-0.035698,-0.196406,-0.031588,0.012295,0,1.0,0.794776,1.0
500,824,VTRS,rw_event,62.0,2024-09-06 14:30:00,2024-09-19 12:30:00,1.0,-0.030541,2024-09-19 12:30:00,2024-09-24 10:30:00,2238.0,-0.042193,-0.249865,-0.033881,0.049729,0,2.0,0.794776,1.0
422,714,RVTY,rw_event,47.0,2024-09-10 15:30:00,2024-09-19 12:30:00,NaN,NaN,None,None,1223.0,-0.054620,-0.336854,-0.041814,0.004721,0,2.0,0.794776,0.0
292,535,MPC,rw_event,52.0,2024-09-10 09:30:00,2024-09-19 11:30:00,1.0,-0.053491,2024-09-19 11:30:00,2024-09-24 10:30:00,3073.0,-0.047008,-0.231289,-0.036579,0.013108,1,1.0,0.794776,1.0
909,18,ALB,hs_event,39.0,2024-09-11 14:30:00,2024-09-19 10:30:00,1.0,-0.016905,2024-09-19 10:30:00,2024-09-24 10:30:00,2440.0,-0.108482,-0.451665,-0.103919,0.034062,1,3.0,0.281046,1.0
78,134,CDNS,rw_event,32.0,2024-09-12 13:30:00,2024-09-19 09:30:00,1.0,-0.005971,2024-09-19 09:30:00,2024-09-24 10:30:00,2389.0,-0.052383,-0.257301,-0.040141,0.048141,0,1.0,0.794776,1.0
1165,81,CFG,ihs_event,53.0,2024-09-06 15:30:00,2024-09-18 11:30:00,1.0,0.041340,2024-09-18 11:30:00,2024-09-24 10:30:00,2644.0,0.062986,-0.030601,0.053380,0.244282,0,1.0,0.790960,1.0
39,62,APD,rw_event,59.0,2024-09-05 09:30:00,2024-09-17 11:30:00,1.0,-0.002195,2024-09-17 11:30:00,2024-09-24 10:30:00,3055.0,-0.038936,-0.235773,-0.027457,0.028657,0,1.0,0.794776,1.0


### Check Results by Event

In [12]:
final4[(final4['event_observations']> 10)&
      (pd.to_datetime(final4['after_event_end_time'])<(datetime.today()-timedelta(days=1)))]. \
        groupby(['event']).Indicator.describe()

,count,mean,std,min,25%,50%,75%,max
event,,,,,,,,
fw_event,375.0,0.717333,0.450897,0.0,0.0,1.0,1.0,1.0
hs_event,213.0,0.305164,0.461562,0.0,0.0,0.0,1.0,1.0
ihs_event,185.0,0.627027,0.484907,0.0,0.0,1.0,1.0,1.0
rw_event,596.0,0.716443,0.451103,0.0,0.0,1.0,1.0,1.0


In [13]:
from tabulate import tabulate
import io
from contextlib import redirect_stdout
df_print = final4.sort_values(['event_end_time'],ascending=0).head(15)
df_print.drop(['index','after_event_observations','after_event_mean','count','mean','min','median','max','Indicator'],axis=1,inplace=True)
df_print = tabulate(df_print, headers='keys', tablefmt="grid")   
frontstring = "```\n"
f = io.StringIO()
with redirect_stdout(f):
    print(f"{frontstring}{df_print}")
out = f.getvalue()
print(out)

ModuleNotFoundError: No module named 'tabulate'

In [14]:
df3[(df3.ticker==ticker)&(df3[event]==1)][['close','date','ticker','rw_event_start_time','rw_event_end_time',event]].head(25)

,close,date,ticker,rw_event_start_time,rw_event_end_time,rw_event
213010,165.270004,2024-02-27 11:30:00,TXN,2024-02-27 11:30:00,2024-03-07 11:30:00,1
213011,165.024994,2024-02-27 12:30:00,TXN,2024-02-27 11:30:00,2024-03-07 11:30:00,1
213012,164.839996,2024-02-27 13:30:00,TXN,2024-02-27 11:30:00,2024-03-07 11:30:00,1
213013,165.259995,2024-02-27 14:30:00,TXN,2024-02-27 11:30:00,2024-03-07 11:30:00,1
213014,164.940002,2024-02-27 15:30:00,TXN,2024-02-27 11:30:00,2024-03-07 11:30:00,1
213015,163.175003,2024-02-28 09:30:00,TXN,2024-02-27 11:30:00,2024-03-07 11:30:00,1
213016,163.679993,2024-02-28 10:30:00,TXN,2024-02-27 11:30:00,2024-03-07 11:30:00,1
213017,162.880005,2024-02-28 11:30:00,TXN,2024-02-27 11:30:00,2024-03-07 11:30:00,1
213018,163.130005,2024-02-28 12:30:00,TXN,2024-02-27 11:30:00,2024-03-07 11:30:00,1
213019,162.979996,2024-02-28 13:30:00,TXN,2024-02-27 11:30:00,2024-03-07 11:30:00,1


In [16]:
def find_cup_and_handle(data):
    close_prices = data["close"].values
    local_min = argrelextrema(close_prices, np.less, order=5)[0]
    local_max = argrelextrema(close_prices, np.greater, order=5)[0]

    if len(local_min) < 2 or len(local_max) < 2:
        return None

    for i in range(len(local_min) - 1):
        left = local_min[i]
        right = local_min[i+1]
        middle = np.argmax(close_prices[left:right]) + left

        # Simple cup shape: two lows and a high between them
        if (close_prices[left] > close_prices[middle] * 1.03 and
            close_prices[right] > close_prices[middle] * 1.03 and
            abs(close_prices[left] - close_prices[right]) / close_prices[middle] < 0.1):
            return (left, middle, right)

    return None

In [19]:
test = find_cup_and_handle(df3)

In [21]:
minmax

,date,close
0,2023-11-03 09:30:00,79.030098
1,2023-11-07 15:30:00,77.909698
2,2023-11-08 11:30:00,76.354515
3,2023-11-09 15:30:00,76.680603
4,2023-11-10 09:30:00,77.051003
...,...,...
524,2025-10-15 09:30:00,154.990005
525,2025-10-16 13:30:00,151.289993
526,2025-10-23 15:30:00,171.600006
527,2025-10-23 15:30:00,171.600006
